# 一、数据来源

## 1、测试样本（HG002 BAM）

HG002 样本 60x 覆盖度的 Illumina HiSeq 短读长比对文件 。
下载指令：

In [ ]:
wget ftp://ftp-trace.ncbi.nlm.nih.gov/giab/ftp/data/AshkenazimTrio/HG002_NA24385_son/NIST_HiSeq_HG002_Homogeneity-10953946/NHGRI_Illumina300X_AJtrio_novoalign_bams/HG002.hs37d5.60x.1.bam

In [ ]:
wget ftp://ftp-trace.ncbi.nlm.nih.gov/giab/ftp/data/AshkenazimTrio/HG002_NA24385_son/NIST_HiSeq_HG002_Homogeneity-10953946/NHGRI_Illumina300X_AJtrio_novoalign_bams/HG002.hs37d5.60x.1.bam.bai

## 2、参考基因组

原论文 HG002 评测基于 GRCh37 (hg19) 版本的参考序列 。

In [ ]:
wget ftp://ftp.1000genomes.ebi.ac.uk/vol1/ftp/technical/reference/phase2_reference_assembly_sequence/hs37d5.fa.gz
gunzip hs37d5.fa.gz
samtools faidx hs37d5.fa

## 3、金标准文件与高置信度区域

原论文明确使用 GIAB NIST Tier 1 v0.6 版本的金标准 ，且仅评估该版本提供的 2.51 Gb 高置信度区域内的变异 。

In [ ]:
# 下载金标准 VCF
wget ftp://ftp-trace.ncbi.nlm.nih.gov/giab/ftp/data/AshkenazimTrio/analysis/NIST_SVs_Integration_v0.6/HG002_SVs_Tier1_v0.6.vcf.gz

In [ ]:
# 下载考试范围 BED
wget ftp://ftp-trace.ncbi.nlm.nih.gov/giab/ftp/data/AshkenazimTrio/analysis/NIST_SVs_Integration_v0.6/HG002_SVs_Tier1_v0.6.bed

# 二、环境搭建

Cue 官方基于 Python 3.7 和 PyTorch 1.5.1 开发 。

在VCF处理阶段建议仅使用Linux原生命令，来避免如bcftools等工具触发的libcrypto动态库依赖崩溃报错。

In [ ]:
conda create -n cue_env python=3.7 -y
conda activate cue_env

pip install torch==1.5.1 torchvision==0.6.1

conda install -c bioconda htslib -y
conda install -c bioconda bcftools -y
conda install -c conda-forge gsl=2.5 -y

git clone https://github.com/popiclab/cue.git cue-master
cd cue-master
conda install -c conda-forge pycocotools -y  #使用 Conda 预编译通道安装 pycocotools，绕开本地 gcc 编译报错


pip install -r install/requirements.txt
rm -rf /mnt/home/ygjx/chenkejin/anaconda3/envs/cue_env/lib/python3.7/site-packages/numpy*
pip install numpy==1.21.6

export PYTHONPATH=${PYTHONPATH}:$(pwd)

pip install truvari==3.2.0

mkdir -p data/models
wget --directory-prefix=data/models/ https://storage.googleapis.com/cue-models/latest/cue.v2.pt
    
    
# 1. 创建新环境并直接安装最新版的 truvari
conda create -n truvari_v4 -c conda-forge -c bioconda truvari -y

# 2. 激活新环境
conda activate truvari_v4

# 3. 验证版本（应该显示 v4.x.x）
truvari version

# 三、运行过程

## 1、构建配置文件

In [ ]:
#创建config/hg002_data.yaml文件并写入下面内容：
bam: "/chenkejin/cue-master/data/hg002_benchmark/HG002.hs37d5.60x.1.bam"
fai: "/chenkejin/cue-master/data/hg002_benchmark/hs37d5.fa.fai"
chr_names: null
#创建config/hg002_model.yaml文件，并写入下面内容：
model_path: "data/models/cue.v2.pt"
out_dir: "output_hg002/"
gpu_ids: [0]
n_cpus: 4
batch_size: 12

## 2、模型推理

In [ ]:
#导出环境变量
export PYTHONPATH=${PYTHONPATH}:$(pwd)
# 前台测试运行（看一眼有没有报错立刻停）
python engine/call.py --data_config config/hg002_data.yaml --model_config config/hg002_model.yaml
# 后台挂起运行（正式跑）
nohup python engine/call.py --data_config config/hg002_data.yaml --model_config config/hg002_model.yaml > cue_run.log 2>&1 &
#可使用tail -f cue_run.log 查看进度，结果默认输出在 config/reports/svs.vcf

## 3、VCF清洗

In [ ]:
# 1. 创建结果输出目录
mkdir -p output_hg002
cp config/reports/svs.vcf output_hg002/hg002_raw.vcf
cd output_hg002

# 2. 提取表头并对变异记录按染色体及坐标进行严格排序
(grep "^#" hg002_raw.vcf; grep -v "^#" hg002_raw.vcf | sort -k1,1V -k2,2n) > hg002_sorted.vcf

# 3. 标准化 bgzip 压缩与建库 (Truvari 评测的必须前置条件)
bgzip -c hg002_sorted.vcf > hg002_final.vcf.gz
tabix -p vcf hg002_final.vcf.gz
cd ..

## 4、金标准预处理

原论文仅针对 大于 5kb 的 DEL (缺失) 进行了深度评估，共有 138 个事件 。

In [ ]:
# 使用 zgrep 提取纯 DEL 金标准 (完美绕过依赖报错)
zgrep "^#" data/hg002_benchmark/HG002_SVs_Tier1_v0.6.vcf.gz > HG002_SVs_Tier1_v0.6_DEL_only.vcf
zgrep -v "^#" data/hg002_benchmark/HG002_SVs_Tier1_v0.6.vcf.gz | grep "SVTYPE=DEL" >> HG002_SVs_Tier1_v0.6_DEL_only.vcf

# 压缩并建索引
bgzip -c HG002_SVs_Tier1_v0.6_DEL_only.vcf > HG002_SVs_Tier1_v0.6_DEL_only.vcf.gz
tabix -p vcf HG002_SVs_Tier1_v0.6_DEL_only.vcf.gz

## 5、评估

In [ ]:
# 运行 Truvari 进行精确打分
truvari bench -b HG002_SVs_Tier1_v0.6_DEL_only.vcf.gz \
              -c output_hg002/hg002_final.vcf.gz \
              -o truvari_results_strict \
              --includebed HG002_SVs_Tier1_v0.6.bed \
              --passonly --refdist 500 --pctsize 0.5 --pctovl 0.5 --pctsim 0 \
              --sizemin 5000 --sizemax 10000000

# 打印最终成绩单
cat truvari_results_strict/summary.txt

# 四、462745N、462745T测试

# 1、数据来源

In [9]:
#A3服务器路径：bam: "/data/share/PancreaticWGS/462745T/462745T.sorted.markdup.BQSR.bam"
#bai："/data/share/PancreaticWGS/462745T/462745T.sorted.markdup.BQSR.bai"
#bam: "/data/share/PancreaticWGS/462745N/462745N.sorted.markdup.BQSR.bam"
#bai: "/data/share/PancreaticWGS/462745N/462745N.sorted.markdup.BQSR.bai"
#reference: "/data/share/PancreaticWGS/Homo_sapiens_assembly38.fasta.fai"

## 2、创建配置文件

cue-master/config/462745N_data.yaml

In [ ]:
bam: "/chenkejin/cue-master/data/462745/462745N/462745N.sorted.markdup.BQSR.bam"
fai: "/chenkejin/cue-master/data/462745/Homo_sapiens_assembly38.fasta.fai"
chr_names: null

cue-master/config/462745N_model.yaml

In [ ]:
model_path: "data/models/cue.v2.pt"
out_dir: "output_462745N/"
gpu_ids: [0]
n_cpus: 10
batch_size: 16

# 3、运行

In [ ]:
nohup python engine/call.py --data_config config/hg002_data.yaml --model_config config/hg002_model.yaml > cue_run.log 2>&1 &

# 4、VCF清洗

In [12]:
mkdir -p output_hg002
cp config/reports/svs.vcf output_462745N/462745N_raw.vcf   #不知道为什么，输出的vcf文件总是在config/reports/svs.vcf
cd output_462745N

(grep "^#" 462745N_raw.vcf; grep -v "^#" 462745N_raw.vcf | sort -k1,1V -k2,2n) > 462745N_sorted.vcf

bgzip -c 462745N_sorted.vcf > 462745N_final.vcf.gz
tabix -p vcf 462745N_final.vcf.gz
cd ..

# 5、金标准预处理（还未进行，462745没有金标准）

# 五、HCC1395

配置环境、运行与最上方的一样，配置文件改改参数就行。注意：A3服务器的bam文件时间戳比相应的bai文件新，且cue生成的一大堆中间文件会自动存储在bam文件目录下，因此需要新建一个文件夹为bam、bai文件创建软链接。然后修改yaml文件里的bam输入路径。

In [ ]:
ln -s /data/share/Genomics_datasets/HCC1395/WGS/WGS_EA_N_1.bwa.dedup.bam ./my_local_WGS_EA_N_1.bam
cp /data/share/Genomics_datasets/HCC1395/WGS/WGS_EA_N_1.bwa.dedup.bam.bai ./my_local_WGS_EA_N_1.bam.bai

N和T数据先分别经过Cue获得vcf结果文件。新建个环境来跑truvari

In [ ]:
# 1. 创建新环境并直接安装最新版的 truvari
conda create -n truvari_v4 -c conda-forge -c bioconda truvari -y

# 2. 激活新环境
conda activate truvari_v4

# 3. 验证版本（应该显示 v4.x.x）
truvari version

利用truvari来获得肿瘤特异性变异，-b设置为正常数据，-c设置为肿瘤数据

In [ ]:
truvari bench \
  -b "/data/chenkejin/cue-master/output_WGS_EA_N_1/WGS_EA_N_1_final.vcf.gz" \
  -c "/data/chenkejin/cue-master/output_WGS_EA_T_1/WGS_EA_T_1_final.vcf.gz" \
  -o "/data/chenkejin/cue-master/result/WGS_EA_1/T-N-4000" \
  -f /data/share/Genomics_datasets/HCC1395/reference_genome/GRCh38/GRCh38.d1.vd1.fa \
  --refdist 500 \
  --pctseq 0 \
  --pctsize 0.7 \
  --pctovl 0 \
  --sizemin 4000 \
  --sizemax -1 \
  --bnddist 100

对金标准文件进行处理

In [ ]:
cd /data/chenkejin/cue-master/

(grep "^#" High_Confidence_Somatic_SV_v1.2_FINAL.vcf; grep -v "^#" High_Confidence_Somatic_SV_v1.2_FINAL.vcf | sort -k1,1V -k2,2n) > gold_standard_sorted.vcf

bgzip -c gold_standard_sorted.vcf > gold_standard_sorted.vcf.gz
tabix -p vcf gold_standard_sorted.vcf.gz

conda activate bio_tools
bcftools reheader \
  -f "/data/chenkejin/cue-master/GRCh38.d1.vd1.fa.fai" \
  gold_standard_sorted.vcf.gz \
  -o gold_standard_ready.vcf.gz

tabix -p vcf gold_standard_ready.vcf.gz

conda activate truvari_v4

truvari评估，加上--passonly参数，获得filtered结果

In [ ]:
truvari bench \
  -b "/data/chenkejin/cue-master/gold_standard_ready.vcf.gz" \
  -c "/data/chenkejin/cue-master/result/WGS_EA_1/T-N-50/fp.vcf.gz" \
  -o "/data/chenkejin/cue-master/result/WGS_EA_1/filtered-50" \
  -f /data/share/Genomics_datasets/HCC1395/reference_genome/GRCh38/GRCh38.d1.vd1.fa \
  --passonly \
  --refdist 500 \
  --pctseq 0 \
  --pctsize 0.7 \
  --pctovl 0 \
  --sizemin 50 \
  --sizemax -1 \
  --bnddist 100

去掉--passonly参数，获得unfiltered结果

In [ ]:
truvari bench \
  -b "/data/chenkejin/cue-master/gold_standard_ready.vcf.gz" \
  -c "/data/chenkejin/cue-master/result/WGS_EA_1/T-N-50/fp.vcf.gz" \
  -o "/data/chenkejin/cue-master/result/WGS_EA_1/unfiltered-50" \
  -f /data/share/Genomics_datasets/HCC1395/reference_genome/GRCh38/GRCh38.d1.vd1.fa \
  --refdist 500 \
  --pctseq 0 \
  --pctsize 0.7 \
  --pctovl 0 \
  --sizemin 50 \
  --sizemax -1 \
  --bnddist 100

用 "/data/chenkejin/cue-master/calc_sv_scores.py" 脚本自动化提取filtered和unfiltered的DEL、DUP、INV、INS、TRA数据，并计算得分情况。

代码如下：

In [ ]:
# -*- coding: gbk -*-
import os
import gzip

def count_svtype(vcf_file, target_type):
    """
    读取 VCF 文件并统计特定 SV 类型的数量（支持 .vcf 和 .vcf.gz）
    """
    # 自动处理文件后缀，兼容 .vcf 和 .vcf.gz
    if not os.path.exists(vcf_file):
        alt_file = vcf_file[:-3] if vcf_file.endswith('.gz') else vcf_file + '.gz'
        if os.path.exists(alt_file):
            vcf_file = alt_file
        else:
            return 0 # 如果文件真不存在，返回 0

    count = 0
    open_func = gzip.open if vcf_file.endswith('.gz') else open
    
    # 针对 TRA (易位) 的特殊处理：同时匹配 TRA 和 BND
    types_to_check = [target_type]
    if target_type == 'TRA':
        types_to_check.extend(['BND'])
        
    with open_func(vcf_file, 'rt') as f:
        for line in f:
            if line.startswith('#'):
                continue
            
            parts = line.split('\t')
            if len(parts) < 8:
                continue
                
            alt = parts[4].upper()
            info = parts[7].upper()
            
            # 匹配逻辑
            is_match = False
            for t in types_to_check:
                # 匹配 INFO 里的 SVTYPE= 或者 ALT 里的 <DEL> 格式
                if f"SVTYPE={t}" in info or f"<{t}>" in alt:
                    is_match = True
                    break
                # 针对 BND (易位) 的标准断点符号匹配，如 A]chr2:1000]
                if target_type == 'TRA' and ('[' in alt or ']' in alt):
                    is_match = True
                    break
            
            if is_match:
                count += 1
                
    return count

def evaluate_directory(dir_path, dir_name):
    """
    计算特定 Truvari 输出目录下的 Precision, Recall 和 F1，并将结果保存到本地
    """
    # 新增：定义一个辅助函数，同时打印并收集文本
    output_lines = []
    def log_and_print(text):
        print(text)
        output_lines.append(text)

    log_and_print(f"\n{'='*70}")
    log_and_print(f"?? 评估报告: {dir_name}")
    log_and_print(f"{'='*70}")
    log_and_print(f"| {'SV_TYPE':<8} | {'TP':<5} | {'FP':<5} | {'FN':<5} | {'Precision':<9} | {'Recall':<9} | {'F1_Score':<9} |")
    log_and_print(f"|{'-'*10}|{'-'*7}|{'-'*7}|{'-'*7}|{'-'*11}|{'-'*11}|{'-'*11}|")
    
    for svtype in ['DEL', 'DUP', 'INV', 'INS', 'TRA']:
        # Truvari 输出的核心文件
        tp_comp_file = os.path.join(dir_path, "tp-comp.vcf.gz")
        fp_file = os.path.join(dir_path, "fp.vcf.gz")
        fn_file = os.path.join(dir_path, "fn.vcf.gz")
        
        # 统计数量
        tp = count_svtype(tp_comp_file, svtype)
        fp = count_svtype(fp_file, svtype)
        fn = count_svtype(fn_file, svtype)
        
        # 计算核心指标 (分母为0时设为0)
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
        
        # 打印并收集格式化结果
        log_and_print(f"| {svtype:<8} | {tp:<5} | {fp:<5} | {fn:<5} | {precision:<9.3f} | {recall:<9.3f} | {f1:<9.3f} |")

    # 新增：将收集到的报告写入当前目录下的 evaluation_report.txt 文件中
    if os.path.exists(dir_path):
        report_path = os.path.join(dir_path, "evaluation_report.txt")
        with open(report_path, 'w', encoding='utf-8') as f:
            f.write('\n'.join(output_lines) + '\n')
        print(f"> 报告已成功保存至: {report_path}")

if __name__ == "__main__":
    # 配置你的两个 Truvari 输出目录
    filtered_dir = "/data/chenkejin/cue-master/result/WGS_EA_1/filtered-4000"
    unfiltered_dir = "/data/chenkejin/cue-master/result/WGS_EA_1/unfiltered-4000"
    
    evaluate_directory(filtered_dir, "严格模式 (Filtered - PASS Only)")
    evaluate_directory(unfiltered_dir, "放宽极限模式 (Unfiltered - ALL Variants)")
    print(f"\n{'='*70}\n")

# 六、集群

In [ ]:
ln -s /mnt/home/ygjx/chenkejin/share_group_folder_ygjx/PDAC_WGS/wgs_197_bams/1866277N/1866277N.sorted.markdup.BQSR.bam /mnt/home/ygjx/chenkejin/cue/cue-master/data_links/my_local_1866277N.bam

In [ ]:
cp /mnt/home/ygjx/chenkejin/share_group_folder_ygjx/PDAC_WGS/wgs_197_bams/1866277N/1866277N.sorted.markdup.BQSR.bai /mnt/home/ygjx/chenkejin/cue/cue-master/data_links/my_local_1866277N.bai

## config/1866277N_data.yaml

In [ ]:
bam: "/mnt/home/ygjx/chenkejin/cue/cue-master/data_links/my_local_1866277N.bam"
fai: "/mnt/home/ygjx/chenkejin/cue/cue-master/Homo_sapiens_assembly38.fasta.fai"
chr_names: null

## config/1866277N_model.yaml

In [ ]:
model_path: "/mnt/home/ygjx/chenkejin/cue/cue-master/data/models/cue.v2.pt"
out_dir: "/mnt/home/ygjx/chenkejin/cue/cue-master/output_1866277N/"
gpu_ids: [0]
n_cpus: 24
batch_size: 12

## run_cue.sh

In [ ]:
#!/bin/bash
#SBATCH --job-name=Cue_WGS
#SBATCH --partition=cu             
#SBATCH --nodes=1                  
#SBATCH --cpus-per-task=32         # 申请 32 核（完美切分大节点）
#SBATCH --mem=120G                 # 申请 120GB 内存（对 773G 来说九牛一毛，但对我们绝对够用）
#SBATCH --output=cue_1866277N.log  
#SBATCH --error=cue_1866277N.log   

# 激活环境
source /mnt/home/ygjx/chenkejin/anaconda3/bin/activate cue_env

# 防断链与无头画图魔法
export PYTHONPATH=${PYTHONPATH}:$(pwd)
export MPLBACKEND=Agg

# 打印一下实际分配的核数，然后开跑
echo "任务开始，系统分配的核数是：$(nproc)"
python engine/call.py --data_config config/1866277N_data.yaml --model_config config/1866277N_model.yaml
echo "正在转移 VCF 结果文件..."
# 将结果挪到你真正想要的输出目录，并顺便改个带有样本名的名字防混淆
mv config/reports/svs.vcf /mnt/home/ygjx/chenkejin/cue/cue-master/cue_197_output/1866277N_raw.vcf
echo "全部任务完美结束！"

### 在“/mnt/home/ygjx/chenkejin/cue/cue-master/”目录下输入 sbatch run_cue.sh 提交作业

### tail -f cue_1866277N.log 实时查看日志

# 在集群上编写了自动运行脚本

In [20]:
# run_pipeline_2.py

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import sys
import time
import subprocess

# ================= 路径配置区 =================
INPUT_BAM_DIR = "/mnt/home/ygjx/chenkejin/share_group_folder_ygjx/PDAC_WGS/wgs_197_bams/"
CUE_BASE_DIR = "/mnt/home/ygjx/chenkejin/cue/cue-master/"
DATA_LINKS_DIR = os.path.join(CUE_BASE_DIR, "data_links")
FINAL_OUTPUT_DIR = os.path.join(CUE_BASE_DIR, "cue_197_output")
SCRIPTS_DIR = os.path.join(CUE_BASE_DIR, "slurm_scripts")
SANDBOX_PARENT_DIR = os.path.join(CUE_BASE_DIR, "config_sandboxes")

for d in [DATA_LINKS_DIR, FINAL_OUTPUT_DIR, SCRIPTS_DIR, SANDBOX_PARENT_DIR]:
    os.makedirs(d, exist_ok=True)

os.chdir(CUE_BASE_DIR)

# ================= 并发控制参数 =================
MAX_CONCURRENT_JOBS = 10  # 维持 10 个节点满血并发

# ================= 获取正在集群里跑的任务名 =================
def get_running_job_names():
    cmd = "squeue -u $USER -h -o '%j'"
    try:
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        return [name.strip() for name in result.stdout.strip().split('\n') if name.strip()]
    except Exception:
        return []

# ================= 获取样本 =================
def get_samples():
    samples = []
    skipped_count = 0
    print("正在扫描样本目录并检查已完成及正在运行的任务...")
    
    running_jobs = get_running_job_names()
    
    for item in os.listdir(INPUT_BAM_DIR):
        folder_path = os.path.join(INPUT_BAM_DIR, item)
        if os.path.isdir(folder_path):
            
            # 跳过 1：最终 VCF 已存在
            final_vcf = os.path.join(FINAL_OUTPUT_DIR, f"{item}_raw.vcf")
            if os.path.exists(final_vcf):
                skipped_count += 1
                continue
                
            # 跳过 2：正躺在集群队列里排队/计算
            job_name = f"Cue_{item}"
            if job_name in running_jobs:
                print(f" ⏳ 样本 [{item}] 正在运行或排队，跳过投递...")
                skipped_count += 1
                continue
                
            bam_file = os.path.join(folder_path, f"{item}.sorted.markdup.BQSR.bam")
            bai_file = os.path.join(folder_path, f"{item}.sorted.markdup.BQSR.bai")
            
            if os.path.exists(bam_file) and os.path.exists(bai_file):
                samples.append({"name": item, "bam": bam_file, "bai": bai_file})
                
    print(f"\n扫描完毕！共跳过了 {skipped_count} 个样本。")
    return sorted(samples, key=lambda x: x["name"])

# ================= 检查当前跑了几个任务 =================
def get_running_jobs_count():
    cmd = f"squeue -u $USER -h | wc -l"
    try:
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        return int(result.stdout.strip())
    except Exception:
        return 0

# ================= 主流程 =================
def main():
    samples = get_samples()
    total_samples = len(samples)
    
    if total_samples == 0:
        print("所有样本均已进入处理状态或处理完毕！")
        sys.exit(0)
        
    print(f"✅ 准备排队处理剩下的 {total_samples} 个样本，最高并发数：{MAX_CONCURRENT_JOBS}...\n")
    
    for i, sample_data in enumerate(samples):
        sample = sample_data["name"]
        
        while True:
            current_jobs = get_running_jobs_count()
            if current_jobs < MAX_CONCURRENT_JOBS:
                break
            sys.stdout.write(f"\r[排队中] 当前已有 {current_jobs} 个任务在跑，[{sample}] 等待空闲槽位...".ljust(80))
            sys.stdout.flush()
            time.sleep(30)
            
        sys.stdout.write(f"\r[投递中] ({i+1}/{total_samples}) 正在配置并提交样本: {sample}".ljust(80) + "\n")
        
        # 1. 软链接
        dst_bam = os.path.join(DATA_LINKS_DIR, f"my_local_{sample}.bam")
        dst_bai = os.path.join(DATA_LINKS_DIR, f"my_local_{sample}.bai")
        for dst in [dst_bam, dst_bai]:
            if os.path.exists(dst) or os.path.islink(dst): os.remove(dst)
        os.symlink(sample_data["bam"], dst_bam)
        os.symlink(sample_data["bai"], dst_bai)
        
        # 2. 为该样本建立专属沙盒目录
        sample_sandbox = os.path.join(SANDBOX_PARENT_DIR, sample)
        os.makedirs(sample_sandbox, exist_ok=True)
        
        # YAML 文件写入专属沙盒
        data_yaml = os.path.join(sample_sandbox, f"{sample}_data.yaml")
        with open(data_yaml, "w", encoding="utf-8", newline='\n') as f:
            f.write(f'bam: "{dst_bam}"\n')
            f.write(f'fai: "{CUE_BASE_DIR}Homo_sapiens_assembly38.fasta.fai"\n')
            f.write(f'chr_names: null\n')
            
        model_yaml = os.path.join(sample_sandbox, f"{sample}_model.yaml")
        output_dir_sample = os.path.join(CUE_BASE_DIR, f"output_{sample}/")
        with open(model_yaml, "w", encoding="utf-8", newline='\n') as f:
            f.write(f'model_path: "{CUE_BASE_DIR}data/models/cue.v2.pt"\n')
            f.write(f'out_dir: "{output_dir_sample}"\n')
            f.write(f'gpu_ids: []\n')
            f.write(f'n_cpus: 24\n')
            f.write(f'batch_size: 12\n')

        # 3. 专属提交脚本设置
        slurm_script = os.path.join(SCRIPTS_DIR, f"run_cue_{sample}.sh")
        final_vcf = os.path.join(FINAL_OUTPUT_DIR, f"{sample}_raw.vcf")
        log_file = os.path.join(CUE_BASE_DIR, f"cue_{sample}.log")
        temp_vcf = os.path.join(sample_sandbox, "reports", "svs.vcf")
        
        slurm_content = (
            "#!/bin/bash\n"
            f"#SBATCH --job-name=Cue_{sample}\n"
            "#SBATCH --partition=cu\n"
            "#SBATCH --nodes=1\n"
            "#SBATCH --cpus-per-task=32\n"
            "#SBATCH --mem=120G\n"
            f"#SBATCH --output={log_file}\n"
            f"#SBATCH --error={log_file}\n"
            "\n"
            "source /mnt/home/ygjx/chenkejin/anaconda3/bin/activate cue_env\n"
            "export PYTHONPATH=${PYTHONPATH}:$(pwd)\n"
            "export MPLBACKEND=Agg\n"
            "\n"
            f"python engine/call.py --data_config config_sandboxes/{sample}/{sample}_data.yaml --model_config config_sandboxes/{sample}/{sample}_model.yaml\n"
            f"mv {temp_vcf} {final_vcf}\n"
            # 清理软链接缓冲池中的巨型缓存索引，防止撑爆磁盘
            f"rm -f {DATA_LINKS_DIR}/*{sample}*.auxindex*\n"
            # 删除了 "rm -rf {sample_sandbox}"，现在你的沙盒及其内部的所有 logs/images 将被永久保留
        )
        
        with open(slurm_script, "w", encoding="utf-8", newline='\n') as f:
            f.write(slurm_content)
            
        # 4. 提交任务
        submit_cmd = f"sbatch {slurm_script}"
        subprocess.run(submit_cmd, shell=True, capture_output=True, text=True)
        time.sleep(2)

    print("\n🎉 所有新样本已投递完毕！正在运行的任务与新任务已经完美汇合。")

if __name__ == "__main__":
    main()

### 在cue/cue-master路径下运行nohup python run_pipeline_2.py > pipeline_manager_2.log 2>&1 &
## tail -f pipeline_manager_2.log  随时查看进度。

# T-N获得胰腺癌的somatic

## 环境配置文件为：/mnt/home/ygjx/chenkejin/cue/cue-master/environment.yml，内容如下：

In [ ]:
name: SURVIVOR_v1.0.3

channels:
  - bioconda
  - conda-forge
  - defaults

dependencies:
  # 基础编译与开发工具
  - make
  - gcc_linux-64
  - gxx_linux-64
  - binutils_linux-64
  - git

  # 基础压缩/开发库
  - zlib
  - bzip2
  - xz
  - libgcc-ng
  - libstdcxx-ng

  - vcftools =0.1.16
  
  # 2. 提供 bcftools 和 htslib (包含 bgzip, tabix)
  - bcftools =1.19
  - htslib =1.19

## 运行下方命令行配置环境

In [ ]:
conda env create -f environment.yml
conda activate SURVIVOR_v1.0.3
conda install -c bioconda survivor -y

## 创建run_survivor_pipeline.py，代码如下：

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import sys
import time
import subprocess

# ================= 1. 路径配置区 =================
TASK_LIST_FILE = "/mnt/home/ygjx/chenkejin/delly/delly-main/task_list.txt"
CUE_VCF_DIR = "/mnt/home/ygjx/chenkejin/cue/cue-master/cue_197_output/"

WORK_DIR = "/mnt/home/ygjx/chenkejin/cue/cue-master/survivor_somatic_run/"
SANDBOX_PARENT_DIR = os.path.join(WORK_DIR, "sandboxes")
SCRIPTS_DIR = os.path.join(WORK_DIR, "slurm_scripts")
FINAL_OUTPUT_DIR = os.path.join(WORK_DIR, "somatic_vcf_output")

SURVIVOR_EXEC = "/mnt/home/ygjx/chenkejin/anaconda3/envs/SURVIVOR_v1.0.3/bin/SURVIVOR"

for d in [SANDBOX_PARENT_DIR, SCRIPTS_DIR, FINAL_OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

MAX_CONCURRENT_JOBS = 80

# ================= 2. 任务雷达 =================
def get_running_job_names():
    cmd = "squeue -u $USER -h -o '%.50j'"
    try:
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        return [name.strip() for name in result.stdout.strip().split('\n') if name.strip()]
    except Exception:
        return []

def get_running_jobs_count():
    cmd = f"squeue -u $USER -h | wc -l"
    try:
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        return int(result.stdout.strip())
    except Exception:
        return 0

# ================= 3. 解析配对任务 =================
def get_paired_tasks():
    tasks = []
    running_jobs = get_running_job_names()
    
    with open(TASK_LIST_FILE, 'r', encoding='utf-8') as f:
        for line in f:
            parts = [p.strip() for p in line.strip().split() if p.strip()]
            if len(parts) == 2:
                normal_id, tumor_id = parts
                task_name = f"{tumor_id}_vs_{normal_id}"
                normal_vcf = os.path.join(CUE_VCF_DIR, f"{normal_id}_raw.vcf")
                tumor_vcf = os.path.join(CUE_VCF_DIR, f"{tumor_id}_raw.vcf")
                
                if not (os.path.exists(normal_vcf) and os.path.exists(tumor_vcf)):
                    continue
                if os.path.exists(os.path.join(FINAL_OUTPUT_DIR, f"{task_name}_somatic.vcf")):
                    continue
                if f"Surv_{tumor_id}" in running_jobs:
                    continue
                    
                tasks.append({
                    "task_name": task_name,
                    "tumor_id": tumor_id,
                    "normal_id": normal_id,
                    "tumor_vcf": tumor_vcf,
                    "normal_vcf": normal_vcf
                })
    return tasks

# ================= 4. 重构的无暇 Bash 逻辑 =================
BASH_TEMPLATE = r"""#!/usr/bin/env bash
#SBATCH --job-name=Surv_[[TUMOR_ID]]
#SBATCH --partition=cu
#SBATCH --nodes=1
#SBATCH --cpus-per-task=4
#SBATCH --mem=16G
#SBATCH --output=[[LOG_FILE]]
#SBATCH --error=[[LOG_FILE]]

set -euo pipefail

source /mnt/home/ygjx/chenkejin/anaconda3/bin/activate SURVIVOR_v1.0.3

TUMOR_VCF="[[TUMOR_VCF]]"
NORMAL_VCF="[[NORMAL_VCF]]"
OUTDIR="[[SANDBOX_DIR]]"
SURVIVOR_EXEC="[[SURVIVOR_EXEC]]"
FINAL_VCF_DEST="[[FINAL_VCF]]"

mkdir -p "${OUTDIR}"/{split,merge,somatic,final}

echo "=== [Step 1] 拯救 TRA & 分箱处理 noTRA ==="
# 1. 强行从源文件捞出所有 TRA，绝不交给 filter 销毁！
grep -E '^#|SVTYPE=TRA' "${TUMOR_VCF}" | vcf-sort > "${OUTDIR}/split/tumor.TRA.vcf"
grep -E '^#|SVTYPE=TRA' "${NORMAL_VCF}" | vcf-sort > "${OUTDIR}/split/normal.TRA.vcf"

bins=("50_100" "101_500" "501_1000" "1001_30000" "30000_plus")
mins=(50 101 501 1001 30000)
maxs=(100 500 1000 30000 1000000000)

for sample_type in "tumor" "normal"; do
    if [[ "${sample_type}" == "tumor" ]]; then input_vcf="${TUMOR_VCF}"; else input_vcf="${NORMAL_VCF}"; fi
    
    for i in "${!bins[@]}"; do
        bin_dir="${OUTDIR}/split/${bins[$i]}"
        mkdir -p "${bin_dir}"
        output_name="${sample_type}_out.vcf"
        
        # 用 2>/dev/null 屏蔽掉那些无意义的 TRA 丢失报错
        "${SURVIVOR_EXEC}" filter "${input_vcf}" NA "${mins[$i]}" "${maxs[$i]}" 0 -1 "${bin_dir}/${output_name}" 2>/dev/null || true
        
        # 只提取非 TRA 的普通变异
        grep -w -v "SVTYPE=TRA" "${bin_dir}/${output_name}" | vcf-sort > "${bin_dir}/${sample_type}.noTRA.vcf"
    done
done

echo "=== [Step 2] 动态合并与纯化 ==="
extract_somatic_pure() {
    merged_vcf=$2; somatic_vcf=$3
    awk '/^#/ {print; next} /SUPP_VEC=10/ {print}' "${merged_vcf}" > "${somatic_vcf}"
}

# 1. 合并孤立的 TRA
list_tra="${OUTDIR}/merge/TRA.list"
echo "${OUTDIR}/split/tumor.TRA.vcf" > "${list_tra}"
echo "${OUTDIR}/split/normal.TRA.vcf" >> "${list_tra}"
"${SURVIVOR_EXEC}" merge "${list_tra}" 1000 1 0 0 0 1000 "${OUTDIR}/merge/TRA.merged.vcf" 2>/dev/null || true
extract_somatic_pure "${list_tra}" "${OUTDIR}/merge/TRA.merged.vcf" "${OUTDIR}/somatic/somatic.TRA.vcf"

# 2. 合并 Large SV
list_large="${OUTDIR}/merge/large.noTRA.list"
echo "${OUTDIR}/split/30000_plus/tumor.noTRA.vcf" > "${list_large}"
echo "${OUTDIR}/split/30000_plus/normal.noTRA.vcf" >> "${list_large}"
"${SURVIVOR_EXEC}" merge "${list_large}" 10000 1 0 0 0 10000 "${OUTDIR}/merge/large.noTRA.merged.vcf" 2>/dev/null || true
extract_somatic_pure "${list_large}" "${OUTDIR}/merge/large.noTRA.merged.vcf" "${OUTDIR}/somatic/somatic.30000_plus.noTRA.vcf"

# 3. 合并 Small SV
small_bins=("50_100" "101_500" "501_1000" "1001_30000")
small_dists=(50 100 500 1000)

for i in "${!small_bins[@]}"; do
    list_small="${OUTDIR}/merge/small.${small_bins[$i]}.noTRA.list"
    echo "${OUTDIR}/split/${small_bins[$i]}/tumor.noTRA.vcf" > "${list_small}"
    echo "${OUTDIR}/split/${small_bins[$i]}/normal.noTRA.vcf" >> "${list_small}"
    
    "${SURVIVOR_EXEC}" merge "${list_small}" "${small_dists[$i]}" 1 0 0 0 "${small_dists[$i]}" "${OUTDIR}/merge/small.${small_bins[$i]}.noTRA.merged.vcf" 2>/dev/null || true
    extract_somatic_pure "${list_small}" "${OUTDIR}/merge/small.${small_bins[$i]}.noTRA.merged.vcf" "${OUTDIR}/somatic/somatic.${small_bins[$i]}.noTRA.vcf"
done

echo "=== [Step 3] 终极合流 ==="
combine_and_sort() {
    out_vcf=$1; shift
    first=1
    for vcf in "$@"; do
        if [[ "${first}" == "1" ]]; then
            cat "${vcf}" > "${out_vcf}"
            first=0
        else
            grep -v "^#" "${vcf}" >> "${out_vcf}" || true
        fi
    done
    vcf-sort "${out_vcf}" > "${out_vcf}.sort" && mv "${out_vcf}.sort" "${out_vcf}"
}

combine_and_sort "${OUTDIR}/final/somatic.ALL.vcf" \
    "${OUTDIR}/somatic/somatic.50_100.noTRA.vcf" \
    "${OUTDIR}/somatic/somatic.101_500.noTRA.vcf" \
    "${OUTDIR}/somatic/somatic.501_1000.noTRA.vcf" \
    "${OUTDIR}/somatic/somatic.1001_30000.noTRA.vcf" \
    "${OUTDIR}/somatic/somatic.30000_plus.noTRA.vcf" \
    "${OUTDIR}/somatic/somatic.TRA.vcf"

mv "${OUTDIR}/final/somatic.ALL.vcf" "${FINAL_VCF_DEST}"
"""

# ================= 5. 调度器 =================
def main():
    tasks = get_paired_tasks()
    total_tasks = len(tasks)
    if total_tasks == 0:
        sys.exit(0)
        
    for i, task in enumerate(tasks):
        task_name = task["task_name"]
        while True:
            if get_running_jobs_count() < MAX_CONCURRENT_JOBS:
                break
            time.sleep(5)
            
        task_sandbox = os.path.join(SANDBOX_PARENT_DIR, task_name)
        os.makedirs(task_sandbox, exist_ok=True)
        
        log_file = os.path.join(WORK_DIR, f"survivor_{task_name}.log")
        final_vcf = os.path.join(FINAL_OUTPUT_DIR, f"{task_name}_somatic.vcf")
        slurm_script = os.path.join(SCRIPTS_DIR, f"run_survivor_{task_name}.sh")
        
        script_content = BASH_TEMPLATE.replace("[[TUMOR_ID]]", task['tumor_id'])
        script_content = script_content.replace("[[LOG_FILE]]", log_file)
        script_content = script_content.replace("[[TUMOR_VCF]]", task['tumor_vcf'])
        script_content = script_content.replace("[[NORMAL_VCF]]", task['normal_vcf'])
        script_content = script_content.replace("[[SANDBOX_DIR]]", task_sandbox)
        script_content = script_content.replace("[[SURVIVOR_EXEC]]", SURVIVOR_EXEC)
        script_content = script_content.replace("[[FINAL_VCF]]", final_vcf)
        
        with open(slurm_script, "w", encoding="utf-8", newline='\n') as f:
            f.write(script_content)
            
        subprocess.run(f"sbatch {slurm_script}", shell=True, capture_output=True)
        time.sleep(0.5)

if __name__ == "__main__":
    main()

## 运行下方命令行运行并查看日志

In [ ]:
nohup python -u run_survivor_pipeline.py > survivor_manager.log 2>&1 &
tail -f survivor_manager.log

### 结果保存在/mnt/home/ygjx/chenkejin/cue/cue-master/survivor_somatic_run/